In [14]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy
import time
import random
import threading

In [15]:
batch_size = 128
learning_rate = 0.0001

In [16]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [17]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.c = None
    def upload(self, delta_ws, delta_cs):
        for v, dw in zip(self.model.variables, delta_ws):
            v.assign_add(dw)
        for c_global, dc in zip(self.c, delta_cs):
            c_global.assign_add(dc)
        return self.model, self.c
    def download(self):
        return self.model, self.c
    def initModel(self, x):
        self.model(x)
        if self.c is None:
            self.c = [tf.Variable(tf.zeros_like(v), trainable=False) for v in self.model.trainable_variables]

In [18]:
def valiAll(index_epoch):
    m, _ = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv.append(va_r2.numpy())

In [19]:
class Node:
    def __init__(self, dsName,freq):
        self.model = MLP()
        self.freq = freq
        dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=23000)
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
        self.c_i = None
    def train(self, idx_epoch):
        # 1. 从服务器下载当前全局模型参数 w_t 和全局控制变量 c
        global_model, c_global = ps.download()
        self.model = copy.deepcopy(global_model)

        # 初始化本地 c_i
        if self.c_i is None:
            self.c_i = [tf.zeros_like(v) for v in self.model.variables]

        # 2. 保存本轮开始时的参数 w_t
        w_before = [tf.identity(v) for v in self.model.variables]

        step = 0
        last_tr_mse = None
        last_tr_rmse = None
        last_tr_mae = None
        last_tr_r2 = None

        # 3. 本地用修正梯度做 K 步更新
        for X, y in self.dataset_train:

            with tf.GradientTape() as tape:
                y_pred = self.model(X)
                tr_mse = tf.reduce_mean(tf.square(y_pred - y))

            grads = tape.gradient(tr_mse, self.model.variables)

            # 修正梯度：g - c_i + c
            corrected_grads = [
                g - ci + cg
                for g, ci, cg in zip(grads, self.c_i, c_global)
            ]

            # 用简单 SGD 更新本地模型参数
            for v, g_corr in zip(self.model.variables, corrected_grads):
                v.assign_sub(learning_rate * g_corr)

            # 记录最后一个 batch 的指标（方便打印）
            last_tr_mse = tr_mse
            last_tr_rmse = tf.sqrt(tr_mse)
            last_tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
            last_tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(
                tf.square(y - tf.reduce_mean(y))
            )
            step += 1
        # 4. 本地更新完成后，计算 Δw_i
        w_after = [tf.identity(v) for v in self.model.variables]
        delta_w = [
            w_a - w_b
            for w_a, w_b in zip(w_after, w_before)
        ]

        # 5. 按 SCAFFOLD 公式更新本地控制变量 c_i，并得到 Δc_i
        #    c_i_new = c_i - c + (1 / (K * η)) * (w_before - w_after)
        K = max(step, 1)  # 防止除零
        scale = 1.0 / (K * learning_rate)

        new_c_i = []
        delta_c_i = []
        for ci_old, c_g, w_b, w_a in zip(self.c_i, c_global, w_before, w_after):
            ci_new = ci_old - c_g + scale * (w_b - w_a)
            new_c_i.append(ci_new)
            delta_c_i.append(ci_new - ci_old)

        # 更新本地 c_i
        self.c_i = new_c_i

        # 6. 把 Δw_i 和 Δc_i 上传给服务器
        global_model, c_global = ps.upload(delta_w, delta_c_i)

        # 7. 打印训练信息 + 记录 r2 + 调用你的验证函数
        if last_tr_mse is not None:
            print("node:{} round:{}".format(self.freq, idx_epoch))
            print("train mse:{} rmse:{} mae:{} r2:{}".format(
                last_tr_mse, last_tr_rmse, last_tr_mae, last_tr_r2
            ))
            r2s[self.freq][idx_epoch] = last_tr_r2.numpy()

In [20]:
r2s = {2.4:{},2.5:{},2.6:{}}
r2sv = []

In [21]:
test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [22]:
ps = ParaServer()
ps.initModel(X_v)

In [23]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [24]:
orders = [0, 1, 2]
turn = [np.array([[26, 104], [178, 312], [344, 464], [520, 600]]), np.array([[0, 94], [149, 223], [319, 433], [464, 580]]), np.array([[32, 151], [155, 248], [270, 354], [378, 502]])]
for i in range(600):
    random.shuffle(orders)
    for j in orders:
        for l, r in turn[j]:
            if l <= i < r:
                nodeList[j].train(i)
    valiAll(i)

node:2.5 round:0
train mse:0.18041740357875824 rmse:0.42475569248199463 mae:0.3409934937953949 r2:-0.514704704284668
mse:0.16951297223567963 rmse:0.41171953082084656 mae:0.3294054865837097 r2:-0.4032083749771118
node:2.5 round:1
train mse:0.13002705574035645 rmse:0.36059263348579407 mae:0.2934998869895935 r2:-0.06943488121032715
mse:0.13679632544517517 rmse:0.36985987424850464 mae:0.2980233430862427 r2:-0.13238394260406494
node:2.5 round:2
train mse:0.11532042175531387 rmse:0.33958861231803894 mae:0.2797979414463043 r2:0.03358769416809082
mse:0.12427440285682678 rmse:0.3525257408618927 mae:0.28532102704048157 r2:-0.028728842735290527
node:2.5 round:3
train mse:0.1083674430847168 rmse:0.3291921019554138 mae:0.2724601924419403 r2:0.09479135274887085
mse:0.11758501827716827 rmse:0.3429067134857178 mae:0.27775999903678894 r2:0.026645123958587646
node:2.5 round:4
train mse:0.10271785408258438 rmse:0.32049626111984253 mae:0.2571589946746826 r2:0.15057313442230225
mse:0.11328727006912231 rmse

In [26]:
for v in r2sv:
    print(v)

-0.40320837
-0.13238394
-0.028728843
0.026645124
0.06222129
0.09311509
0.11755329
0.13512182
0.15083289
0.16205001
0.17753083
0.18861854
0.19530857
0.20383048
0.21122587
0.21934569
0.2270872
0.23295468
0.23809183
0.24091297
0.24775243
0.25311762
0.25647837
0.25994885
0.26458108
0.26869422
0.27671695
0.28778338
0.29695207
0.3048094
0.3121807
0.31595147
0.3259158
0.33857578
0.347982
0.35568702
0.35802156
0.3648532
0.37075758
0.3750326
0.37750375
0.38189083
0.38470304
0.38460433
0.38865608
0.3923754
0.392699
0.39415318
0.395963
0.39931756
0.40054244
0.40297788
0.40301174
0.40512353
0.40528107
0.40698582
0.40813088
0.41018516
0.41205674
0.41313523
0.41117096
0.41464382
0.41598266
0.41127956
0.41760784
0.41900384
0.41831034
0.42069775
0.42106694
0.4222203
0.42123884
0.42290395
0.4240954
0.42469132
0.4249578
0.42469263
0.4262231
0.42565447
0.4262507
0.42888618
0.42888403
0.42976183
0.42940748
0.43044496
0.43060726
0.4314562
0.43038905
0.43260247
0.43351233
0.43316305
0.4335758
0.43481737
0.4